# Дневные и недельные метрики ПРАЙМ

Ноутбук читает учебную базу через настройки команды в `.env` и использует назначенный твоему логину сегмент клиентов. Дата отчёта 10 июня 2026 года. Последний день данных — 9 июня UTC. Измени `report_date`, чтобы выбрать другую дату.

In [ ]:
from pathlib import Path
import sys
from datetime import date, timedelta

import pandas as pd

notebooks_dir = Path.cwd() / "notebooks"
if notebooks_dir.is_dir() and str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

from prime.metrics import period_metrics
from prime.report import METRICS
from prime_analytics_data import load_churn_cohorts, load_daily

report_date = date(2026, 6, 10)
day = report_date - timedelta(days=1)
week_start = report_date - timedelta(days=7)

daily, ended_by_channel = load_daily(week_start, report_date, subscriptions=True)
cohorts = load_churn_cohorts([(day, report_date), (week_start, report_date)])
daily[["payments_count", "total_revenue", "active_subs_eod", "ended_subs", "new_subscriptions",
       "payment_quality", "subscription_quality"]]

Выручку и число платежей суммируем. Средний платёж делит общую выручку на число платежей. Снимок активных подписок берём перед следующей полуночью UTC. Число завершений по `ended_at` суммируем.

Новые подписки считаем по `started_at`. Доля оттока по Л5 показывает, какая часть подписок, активных на начало периода, завершилась до его конца. Подписки, оформленные внутри периода, в эту когорту не входят.

Новых платящих считаем по первой успешной оплате владельца за доступную историю. Долю неуспешных списаний считаем по суммам попыток за период. Возвращённый платёж включается в знаменатель как ранее прошедшее списание.

In [ ]:
day_metrics = period_metrics(
    daily, day, report_date, churn_cohort=cohorts[(day, report_date)]
)
week_metrics = period_metrics(
    daily, week_start, report_date, churn_cohort=cohorts[(week_start, report_date)]
)
results = pd.DataFrame({
    label: {name: metrics[name] for name in METRICS}
    for label, metrics in (("день", day_metrics), ("7 дней", week_metrics))
})
formatted = results.astype(object)
for name, (_, kind) in METRICS.items():
    if kind == "percent":
        for label in results.columns:
            rate = results.loc[name, label]
            formatted.loc[name, label] = f"{rate:.2%}" if pd.notna(rate) else "—"
formatted = formatted.rename(index={key: name for key, (name, _) in METRICS.items()})
display(formatted)

for label, metrics in (("день", day_metrics), ("7 дней", week_metrics)):
    issues = {key: details for key, details in metrics["issues"].items() if key in METRICS}
    if issues:
        print(f"Причины пропуска за {label}:", issues)

In [ ]:
if report_date == date(2026, 6, 10):
    print("Сверка с первым прогоном загрузчика для твоего сегмента")
    expected = {"revenue": 369562, "payments_count": 1138}
    display(pd.DataFrame({"из базы": results["день"], "первый прогон": expected}).loc[list(expected)])

last_day_channels = ended_by_channel.loc[
    ended_by_channel["day"].eq(pd.Timestamp(day)), ["channel", "ended_subs"]
]
display(last_day_channels)

`payment_quality` показывает причину пропуска платёжных метрик. Проверяем наличие платежей, покрытие начала и конца дня UTC, наличие сессий и положительные суммы успешных платежей.

`subscription_quality` сообщает, сошлось ли число завершений с суммой по каналам и были ли в этот день завершения во всём источнике. Если их нет, показываем состояние данных как неопределённое. Метрики подписок за период с таким днём остаются пустыми, хотя в дневной витрине видны исходные количества. Статус `reconciled` означает сверку счёта и наличие событий в источнике. Для гарантии полноты нужен отдельный контроль загрузки.